# Extract Data

In [2]:
import fitz  # pip install pymupdf
import pandas as pd

# Open the PDF
doc = fitz.open(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\Kenya-Demographic-and-Health-Survey-2022-Factsheet-Embu.pdf")

# Extract all text page by page
full_text = ""
for page in doc:
    full_text += page.get_text()

print(full_text)

2022 Kenya DHS Factsheet - Embu 
 
2022  
Kenya Demographic and Health Survey  
Fact Sheet 
Embu County 
 
Characteristics of Households and Respondents 
Embu 
Kenya 
Household population with access to at least basic drinking water service (%) 
73 
68  
Household population with at least basic sanitation service (%) 
48 
41  
Household population relying on clean fuels and technologies for cooking, space heating, & lighting (%) 
15 
21  
Women age 15-49 with no formal education (%) 
1 
6  
Men age 15-49 with no formal education (%) 
 <1 
3  
Fertility and Family Planning (FP) 
 
 
Total fertility rate (number of children per woman) 
3.1 
3.4  
Teenage pregnancy (% age 15-19 who have ever been pregnant) 
14 
15  
Use of modern method of FP (% of married women age 15-49) 
75 
57  
Unmet need for FP1 (% of married women age 15-49) 
2 
14  
Demand for FP satisfied by modern methods (% of married women age 15-49) 
89 
75  
Maternal and Child Health  
 
 
Births delivered by a skilled provi

In [3]:
import pdfplumber

with pdfplumber.open(r"C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\data\Kenya-Demographic-and-Health-Survey-2022-Factsheet-Embu.pdf") as pdf:
    all_tables = []
    for page in pdf.pages:
        all_tables.extend(page.extract_tables())

In [11]:
raw = all_tables[1]  # second table

# Merge continuation rows (where row[0] is None) into the previous row's indicator
merged_rows = []
for row in raw[1:]:  # skip header row
    if row[0] is None and merged_rows:
        prev = merged_rows[-1]
        extra = " ".join(str(x) for x in row if x)
        prev["indicator"] = (prev["indicator"] + " " + extra).strip()
    else:
        indicator = (row[1] or "").strip()
        embu_val  = (row[4] or row[3] or "").strip()
        kenya_val = (row[7] or row[6] or "").strip()
        merged_rows.append({"indicator": indicator, "embu": embu_val, "kenya": kenya_val})

# Tag section headers (rows where both values are empty)
section = ""
records = []
for r in merged_rows:
    if not r["embu"] and not r["kenya"]:
        section = r["indicator"]
    else:
        records.append({"section": section, "indicator": r["indicator"], "embu": r["embu"], "kenya": r["kenya"]})

embu_dhs = pd.DataFrame(records)
embu_dhs


,section,indicator,embu,kenya
0,,Household population with access to at least b...,73,68
1,,Household population with at least basic sanit...,48,41
2,,Household population relying on clean fuels an...,15,21
3,,Women age 15-49 with no formal education (%),1,6
4,,Men age 15-49 with no formal education (%),<1,3
5,Fertility and Family Planning (FP),Total fertility rate (number of children per w...,3.1,3.4
6,Fertility and Family Planning (FP),Teenage pregnancy (% age 15-19 who have ever b...,14,15
7,Fertility and Family Planning (FP),Use of modern method of FP (% of married women...,75,57
8,Fertility and Family Planning (FP),Unmet need for FP1 (% of married women age 15-49),2,14
9,Fertility and Family Planning (FP),Demand for FP satisfied by modern methods (% o...,89,75


In [13]:
embu_dhs_pivot = embu_dhs.set_index('indicator')[['embu', 'kenya']].T
embu_dhs_pivot.to_csv("embu_dhs_2022.csv")

In [14]:
import time
t = time.time()
import pandas as pd
df = pd.read_csv(r'C:\Users\Mercy\Documents\Tendri\Snowflake Pulls\Xana\snowflake\tendri\embu_dhs_2022.csv')
print(f"Done in {time.time()-t:.2f}s, shape: {df.shape}")


Done in 0.01s, shape: (2, 32)
